# 11 - Analise de Dataset Completa

## Objetivo
Aplicar todos os conceitos em uma analise integrada de um dataset.

## Contexto
Vamos analisar um dataset de vendas ficticio com dados de
vendedores, regioes, produtos, quantidades e precos. O objetivo
e responder perguntas de negocio usando Pandas.

## Etapas de uma analise
1. Carregar e inspecionar os dados.
2. Limpar: tratar NaN, duplicatas e tipos.
3. Explorar com estatisticas descritivas.
4. Agrupar por dimensoes de interesse.
5. Combinar tabelas quando necessario.
6. Criar variaveis derivadas.
7. Extrair insights e resumir.

## Perguntas de negocio
- Qual o faturamento total e por regiao?
- Qual o vendedor com melhor desempenho?
- Qual o produto mais vendido em quantidade?
- Como as vendas evoluem ao longo do tempo?
- Existe correlacao entre preco unitario e quantidade?

## Conceitos aplicados
- Criacao de DataFrame com NumPy.
- Tratamento de NaN e duplicatas.
- Groupby com multiplas agregacoes.
- Merge entre tabelas.
- Ordenacao e ranking.
- Estatisticas por grupo.
- Series temporais com reamostragem mensal.
- Tabela dinamica.

## 1. Criacao do dataset sintetico
Geramos 500 registros de vendas aleatorias com `numpy.random` (semente
fixa para reprodutibilidade). Em seguida, **introduzimos problemas de
qualidade** de proposito — NaN em `preco_unitario` e linhas duplicadas —
para praticar a etapa de limpeza.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# 1. Criando dataset sintetico de vendas
n = 500
datas = pd.date_range("2023-01-01", periods=n, freq="D")
vendedores = ["Ana", "Bruno", "Carla", "Diego", "Elisa"]
regioes = ["Norte", "Sul", "Leste", "Oeste"]
produtos = ["A", "B", "C", "D"]

vendas = pd.DataFrame({
    "data": np.random.choice(datas, n),
    "vendedor": np.random.choice(vendedores, n),
    "regiao": np.random.choice(regioes, n),
    "produto": np.random.choice(produtos, n),
    "quantidade": np.random.randint(1, 20, n),
    "preco_unitario": np.random.uniform(10, 200, n).round(2),
})

# Inserindo problemas de qualidade
vendas.loc[vendas.sample(20, random_state=1).index, "preco_unitario"] = np.nan
vendas = pd.concat([vendas, vendas.sample(10, random_state=2)],
                   ignore_index=True)
vendas = vendas.sample(frac=1, random_state=3).reset_index(drop=True)

print("Shape inicial:", vendas.shape)
print("\nHead:\n", vendas.head())
print("\nInfo:")
vendas.info()

## 2. Limpeza dos dados
Tres etapas de limpeza:
1. **Duplicatas**: remover linhas repetidas.
2. **Tipos**: converter `data` para `datetime` e colunas categoricas
   (`vendedor`, `regiao`, `produto`) para `category` (economia de memoria).
3. **NaN**: preencher `preco_unitario` com a mediana (robusta a outliers).

Por fim, criamos a variavel derivada `faturamento = quantidade * preco_unitario`.

In [ ]:
# 2. Limpeza

# Duplicatas
print("\nDuplicatas:", vendas.duplicated().sum())
vendas = vendas.drop_duplicates().reset_index(drop=True)
print("Shape apos remover duplicatas:", vendas.shape)

# Tipos
vendas["data"] = pd.to_datetime(vendas["data"])
vendas["vendedor"] = vendas["vendedor"].astype("category")
vendas["regiao"] = vendas["regiao"].astype("category")
vendas["produto"] = vendas["produto"].astype("category")

# NaN em preco
print("\nNaN em preco_unitario:", vendas["preco_unitario"].isna().sum())
vendas["preco_unitario"] = vendas["preco_unitario"].fillna(
    vendas["preco_unitario"].median()
)

# Variavel derivada: faturamento
vendas["faturamento"] = (vendas["quantidade"]
                         * vendas["preco_unitario"]).round(2)

print("\nShape final apos limpeza:", vendas.shape)
print("NaN restantes:", vendas.isna().sum().sum())
print("Dtypes:\n", vendas.dtypes)

## 3. Estatisticas gerais
`describe()` nas colunas numericas para ter uma visao geral da
distribuicao: quantidade, preco unitario e faturamento.

In [ ]:
# 3. Estatisticas gerais
print("\nDescribe:\n", vendas[["quantidade",
                                "preco_unitario",
                                "faturamento"]].describe())

## 4. Analises por dimensao
Agrupamos por diferentes dimensoes para responder as perguntas de negocio:
- **Faturamento total** da empresa.
- **Por regiao**: soma, media e contagem.
- **Por vendedor**: soma, media e contagem.
- **Por produto**: quantidade total e faturamento total.

In [ ]:
# 4. Analises por dimensao

# Faturamento total
print("\nFaturamento total:", round(vendas["faturamento"].sum(), 2))

# Por regiao
por_regiao = (vendas.groupby("regiao", observed=True)["faturamento"]
              .agg(["sum", "mean", "count"])
              .round(2)
              .sort_values("sum", ascending=False))
print("\nFaturamento por regiao:\n", por_regiao)

# Por vendedor
por_vendedor = (vendas.groupby("vendedor", observed=True)["faturamento"]
                .agg(["sum", "mean", "count"])
                .round(2)
                .sort_values("sum", ascending=False))
print("\nFaturamento por vendedor:\n", por_vendedor)

# Produto mais vendido em quantidade
por_produto = (vendas.groupby("produto", observed=True)
               .agg(quantidade_total=("quantidade", "sum"),
                    faturamento_total=("faturamento", "sum"))
               .round(2)
               .sort_values("quantidade_total", ascending=False))
print("\nPor produto:\n", por_produto)

## 5. Merge com tabela de metas
Combinamos as vendas com uma tabela de metas por vendedor (merge `left`).
Depois calculamos o **percentual de atingimento** = faturamento / meta * 100.

In [ ]:
# 5. Merge com tabela de metas
metas = pd.DataFrame({
    "vendedor": ["Ana", "Bruno", "Carla", "Diego", "Elisa"],
    "meta": [50000, 60000, 45000, 70000, 55000],
})
vendas_com_meta = pd.merge(vendas, metas, on="vendedor", how="left")

# Atingimento por vendedor
atingimento = (vendas_com_meta.groupby("vendedor", observed=True)
               .agg(faturamento=("faturamento", "sum"),
                    meta=("meta", "first"))
               .assign(atingimento=lambda d: (d["faturamento"]
                                              / d["meta"] * 100).round(2))
               .sort_values("atingimento", ascending=False))
print("\nAtingimento de meta:\n", atingimento)

## 6. Evolucao temporal
Usamos `resample("ME")` (month end) para agregar o faturamento por mes.
Isso exige o indice do DataFrame definido como a coluna `data`.

In [ ]:
# 6. Evolucao temporal
vendas_mes = (vendas.set_index("data")
              .resample("ME")["faturamento"]
              .sum()
              .round(2))
print("\nFaturamento mensal:\n", vendas_mes)

## 7. Correlacao entre quantidade e preco
Calculamos a correlacao de Pearson entre `quantidade` e `preco_unitario`
para investigar se compradores preferem produtos mais baratos ou nao.

In [ ]:
# 7. Correlacao
print("\nCorrelacao quantidade x preco:\n",
      vendas[["quantidade", "preco_unitario"]].corr().round(4))

## 8. Tabela dinamica (regiao x produto)
`pivot_table` cruza `regiao` (linhas) com `produto` (colunas) e soma o
faturamento. Util para visualizar rapidamente a distribuicao cruzada.

In [ ]:
# 8. Tabela dinamica
pivot = pd.pivot_table(vendas, values="faturamento",
                       index="regiao", columns="produto",
                       aggfunc="sum", observed=True).round(2)
print("\nPivot regiao x produto:\n", pivot)

## 9. Top N
`head(3)` sobre um DataFrame ja ordenado retorna o top 3 por faturamento.

In [ ]:
# 9. Top N
print("\nTop 3 vendedores por faturamento:\n",
      por_vendedor.head(3))

## 10. Resumo final
Consolidamos os principais numeros da analise em um bloco de texto
formatado, respondendo diretamente as perguntas de negocio.

In [ ]:
# 10. Resumo final
print("\nResumo da analise:")
print(f"  Total de vendas: {len(vendas)}")
print(f"  Faturamento total: R$ {vendas['faturamento'].sum():,.2f}")
print(f"  Ticket medio: R$ {vendas['faturamento'].mean():,.2f}")
print(f"  Melhor vendedor: {por_vendedor.index[0]}")
print(f"  Melhor regiao: {por_regiao.index[0]}")
print(f"  Produto mais vendido: {por_produto.index[0]}")